# ReliablePipeline: the reliability-aware prediction lifecycle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/pipeline.ipynb)

Companion notebook for the ReliableML paper `docs/latex/pipeline.tex`. It walks through fit, predict with a reliability report and plots, persistence, the `reliability_required` switch, and how the labeled analyses (evaluation, deterioration, explanation, reason audit) plug into the same object.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Design

`ReliablePipeline(estimator, ...)` wraps any scikit-learn-style estimator and a `DriftMonitor`:

1. **`fit(X, y)`** fits the monitor's reference profiles (quantile bins, categories, moments, and the label-conditional distributions) *and* the estimator, and writes `training-distributions.png` to `plot_dir`.
2. **`predict(X)`** returns ordinary predictions, and as a side effect compares the batch with the reference and writes `prediction-<timestamp>.png/.json` (the report, predictions and output type). The latest report is in `last_report`, the file paths in `last_plot_path` and `last_report_path`.
3. **`predict_with_report(X)`** returns `(predictions, report)`.

The report's display score is $S=\exp(-\overline{\mathrm{PSI}})$ over features, with per-feature PSI, KL, KS, Wasserstein, DSI and chi-square statistics (see the distribution-shift notebook). It is descriptive: a **similarity-to-training** summary, not a probability that predictions are correct.

| Analysis | Method | Data requirement |
|---|---|---|
| Feature shift | `assess_covariate_shift(X)` | unlabeled batch and fitted reference |
| Concept shift | `assess_concept_shift(X, y)` | labeled batch |
| Classification evaluation | `evaluate(X, y_true, ...)` | labeled representative batch |
| Fairness | `evaluate(..., sensitive_features=...)` | labels, groups, positive class |
| Deterioration | `assess_deterioration(history)` | $\ge3$ dated labeled batches |
| Explanation | `explain(X, class_label=...)` | background data and target class |
| Reason audit | `audit_reasons(X, y_true, ...)` | labeled holdout, reference background |

## 2. Fit and predict

In [ ]:
import json, os, tempfile, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.image as mpimg
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ml_reliability import ReliablePipeline, assess_deterioration

warnings.filterwarnings("ignore")
bc = load_breast_cancer(as_frame=True)
cols = ["mean radius", "mean texture", "mean smoothness", "mean compactness", "mean symmetry", "mean fractal dimension"]
X, y = bc.data[cols], bc.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)

pipe = ReliablePipeline(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                        plot_dir="reliability_plots", random_seed=0).fit(Xtr, ytr)
print("files after fit:", sorted(os.listdir("reliability_plots")))

predictions, report = pipe.predict_with_report(Xte)
print("files after predict:", sorted(os.listdir("reliability_plots")))
print("in-domain batch -> similarity", round(report.similarity_score, 3), "| mean PSI", round(report.mean_psi, 4),
      "| covariate shift flagged:", report.covariate_shift_detected, "| drifted:", list(report.drifted_features))

In [ ]:
# The artifacts the pipeline wrote on the last predict()
payload = json.loads(pipe.last_report_path.read_text())
print("report JSON keys:", list(payload))
print("output_type:", payload["output_type"], "| first predictions:", payload["predictions"][:5])
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(pipe.last_plot_path)); ax.axis("off"); ax.set_title(pipe.last_plot_path.name)
plt.show()

### A shifted batch vs. an in-domain batch

In [ ]:
shifted = Xte.copy()
shifted["mean radius"] = shifted["mean radius"] * 1.5 + 2.0   # e.g. a unit/instrument change
shifted["mean texture"] = shifted["mean texture"] + 6.0
for name, batch in [("in-domain", Xte), ("shifted", shifted)]:
    rep = pipe.assess_covariate_shift(batch)
    print(f"{name:10s} similarity={rep.similarity_score:.3f} mean PSI={rep.mean_psi:.3f} flagged={rep.covariate_shift_detected} drifted={list(rep.drifted_features)}")
print("accuracy on in-domain:", round((pipe.predict(Xte) == yte).mean(), 3), "| on shifted:", round((pipe.predict(shifted) == yte).mean(), 3),
      "(a shift report is a warning to check outcomes, not a measure of accuracy)")
concept = pipe.assess_concept_shift(Xte, yte)
print("concept shift (labels as in training):", "mean JS =", round(concept.mean_js_divergence, 4), "| shifted:", list(concept.shifted_features))

## 3. Persistence, schema checks and the `reliability_required` switch

`save`/`load` use **pickle**: only load files you created or fully trust, with the same library and scikit-learn versions. Batches must have the same columns as training.

In [ ]:
with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "pipeline.pkl")
    pipe.save(path)
    restored = ReliablePipeline.load(path)
    print("round trip predictions identical:", bool((restored.predict(Xte) == predictions).all()))

try:
    pipe.predict(Xte.drop(columns=["mean radius"]))
except Exception as error:
    print("missing column ->", type(error).__name__ + ":", str(error)[:120])

plain = ReliablePipeline(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                         reliability_required=False, plot_dir="reliability_plots_off").fit(Xtr, ytr)
print("reliability_required=False: predictions work:", plain.predict(Xte)[:3], "| last_report:", plain.last_report)
for call in (lambda: plain.assess_covariate_shift(Xte), lambda: plain.predict_with_report(Xte)):
    try:
        call()
    except RuntimeError as error:
        print("  ->", str(error)[:90])

## 4. Labeled analyses on the same object

Each call below is a one-liner on the fitted pipeline; their settings are kept small here for speed. The dedicated notebooks (evaluation, fairness, deterioration, explainability, reason-audit) explain the outputs in detail.

In [ ]:
size = np.where(Xte["mean radius"] > Xte["mean radius"].median(), "large", "small")  # illustrative group, not protected
fast = ReliablePipeline(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), plot_dir="reliability_plots",
                        bias_variance_folds=3, bias_variance_repeats=3, random_seed=0).fit(Xtr, ytr)
ev = fast.evaluate(Xte, yte, sensitive_features={"size": size}, positive_label=1).to_dict()
cls = ev["classification"]
print("evaluate  -> accuracy", round(cls["accuracy"], 3), "| MCC", round(cls["matthews_correlation_coefficient"], 3),
      "| ECE", round(cls["probability_calibration"]["top_label_expected_calibration_error"], 3),
      "| parity gap", round(ev["fairness"]["size"]["classes"]["1"]["statistical_parity"]["gap"], 3))

batches = np.array_split(np.argsort(Xte["mean texture"].to_numpy()), 4)  # pseudo-batches, NOT real time
history = [{"timestamp": k, "y_true": yte.iloc[i].to_numpy(), "y_pred": fast.estimator.predict(Xte.iloc[i])} for k, i in enumerate(batches)]
det = fast.assess_deterioration(history, bootstrap_samples=200)
print("deterioration -> slope", round(det.accuracy_slope, 4), "| flagged:", det.deterioration_detected, "(pseudo-batches)")

exp = fast.explain(Xte.iloc[:20], class_label=1, background_data=Xtr.sample(50, random_state=0), max_instances=10, num_samples=100)
print("explain   -> top feature:", exp.global_shap_importance[0]["feature"], "| alignment:", round(exp.reason_alignment["alignment_score_0_to_1"], 3))

aud = fast.audit_reasons(Xte.iloc[:120], yte.iloc[:120], positive_label=1, bootstrap_samples=100, explanation_samples=100, max_explained_instances=10)
top = max(aud.feature_reliance, key=lambda r: r["loss_increase"])
print("audit     -> most relied-on feature:", top["feature"], "| loss increase", round(top["loss_increase"], 4))

## Interpretation and limitations

**How to read the output.** A reliability report summarizes how a new batch compares with the training reference; it appears (with plots and JSON) on every `predict`, so monitoring is automatic and leaves an audit trail. The shifted example shows a flagged batch, and the accuracy comparison reminds you that the shift report does not measure correctness - labeled evaluation, deterioration and the reason audit answer that when outcomes become available.

**Limitations.** The reference is the training data only; reports are per batch with sampling noise at small batch sizes; thresholds (PSI 0.2 etc.) are conventions to be calibrated; the pipeline writes files on every predict (set `plot_dir` deliberately and treat reports as potentially sensitive); pickled pipelines are not a safe interchange format; supported estimators must be scikit-learn compatible, and the explanation and audit tools additionally need `predict_proba`, `classes_` and finite numeric features.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.